# Day 3: Vector Databases, Walked Through

This notebook ties together the whole week: documents (Day 1), embeddings (Day 2),
and now storing + searching those embeddings at scale (Day 3).

Run each cell in order. No installs needed -- plain Python throughout.

## Cell 1: What is a vector database?

A vector database stores embeddings (lists of numbers) along with metadata, and
lets you quickly find the ones most similar to a new embedding.

It's not magic -- it's a specialized search index, the same way a normal database
is specialized for finding rows by ID. A vector database is specialized for finding
vectors that are *close* to each other.

## Cell 2: Build a simple version

Let's build one from scratch. It needs two things: a place to store (embedding,
metadata) pairs, and a way to search them by similarity.

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(y * y for y in b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)

class SimpleVectorDB:
    def __init__(self):
        self.entries = []

    def add(self, embedding, metadata=None):
        self.entries.append({"embedding": embedding, "metadata": metadata or {}})

    def search(self, query_embedding, top_k=3):
        scored = [
            (entry, cosine_similarity(query_embedding, entry["embedding"]))
            for entry in self.entries
        ]
        scored.sort(key=lambda pair: pair[1], reverse=True)
        return [{"metadata": e["metadata"], "similarity": s} for e, s in scored[:top_k]]

db = SimpleVectorDB()
db.add([0.9, 0.8, 0.1, 0.2], {"word": "cat"})
db.add([0.8, 0.9, 0.2, 0.1], {"word": "dog"})
db.add([0.1, 0.2, 0.9, 0.8], {"word": "car"})

results = db.search([0.88, 0.82, 0.12, 0.18], top_k=2)
for r in results:
    print(r["metadata"]["word"], round(r["similarity"], 3))

This is BRUTE FORCE search -- it checks every stored vector, every time. Fine for a
handful of items. Not fine for a million.

## Cell 3: Implement indexing

Real vector databases avoid checking everything by building an index ahead of time.
Here's a simplified version inspired by HNSW: connect each vector to a handful of
its nearby "neighbors", then search by hopping toward better matches instead of
checking every single vector.

In [ ]:
import random

def make_random_vectors(count, dimensions=8, seed=42):
    rng = random.Random(seed)
    return [[rng.random() for _ in range(dimensions)] for _ in range(count)]

def brute_force_search(query, vectors, top_k=5):
    scored = [(i, cosine_similarity(query, v)) for i, v in enumerate(vectors)]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]

class SimpleGraphIndex:
    def __init__(self, vectors, num_neighbors=20, seed=42):
        self.vectors = vectors
        self.num_neighbors = num_neighbors
        self.graph = self._build_graph(seed)

    def _build_graph(self, seed):
        rng = random.Random(seed)
        graph = {}
        for i, vector in enumerate(self.vectors):
            sample_size = min(150, len(self.vectors))
            candidates = rng.sample(range(len(self.vectors)), sample_size)
            scored = [(j, cosine_similarity(vector, self.vectors[j])) for j in candidates if j != i]
            scored.sort(key=lambda pair: pair[1], reverse=True)
            graph[i] = [j for j, _ in scored[:self.num_neighbors]]
        return graph

    def search(self, query, top_k=5, entry_points=10, max_steps=40, beam_width=10):
        rng = random.Random()
        visited = set()
        scored = {}
        frontier = rng.sample(range(len(self.vectors)), min(entry_points, len(self.vectors)))
        for _ in range(max_steps):
            if not frontier:
                break
            for idx in frontier:
                if idx not in visited:
                    visited.add(idx)
                    scored[idx] = cosine_similarity(query, self.vectors[idx])
            beam = sorted(scored.items(), key=lambda pair: pair[1], reverse=True)[:beam_width]
            next_frontier = [n for idx, _ in beam for n in self.graph[idx] if n not in visited]
            if not next_frontier:
                break
            frontier = next_frontier
        ranked = sorted(scored.items(), key=lambda pair: pair[1], reverse=True)
        return ranked[:top_k], len(visited)

print("Index built. Ready to compare speed in the next cell.")

## Cell 4: Show benchmarks

Let's time brute force vs. the graph index on a few dataset sizes, and see the gap
grow. (Using smaller sizes here than the full benchmark.py so this cell runs fast.)

In [ ]:
import time

for size in [100, 1000, 5000]:
    vectors = make_random_vectors(size, seed=42)
    query = make_random_vectors(1, seed=999)[0]

    start = time.perf_counter()
    brute_top = brute_force_search(query, vectors, top_k=5)
    brute_ms = (time.perf_counter() - start) * 1000

    index = SimpleGraphIndex(vectors)
    start = time.perf_counter()
    graph_top, visited = index.search(query, top_k=5)
    graph_ms = (time.perf_counter() - start) * 1000

    print(f"{size:>6,} vectors | brute: {brute_ms:7.3f} ms | graph: {graph_ms:7.3f} ms (checked {visited}) | top match same: {brute_top[0][0] == graph_top[0][0]}")

Brute force time grows with dataset size. The graph index stays fast because it only
checks a small, targeted slice of the data -- that's the entire point of indexing.

## Cell 5: Compare real vector databases

Our SimpleVectorDB and SimpleGraphIndex are toy versions. Real vector databases
(Pinecone, Milvus, Weaviate, Chroma, FAISS) do the same two jobs -- store vectors,
search by similarity -- but with production-grade indexing, persistence, and scale.

| Database | Hosting | Best for |
|---|---|---|
| Pinecone | Managed cloud | Fast setup, no infra to run |
| Milvus | Self-hosted (Docker) | Massive scale, full control |
| Weaviate | Self-hosted or managed | Hybrid keyword + vector search |
| Chroma | Local / in-process | Prototyping, small projects |
| FAISS | Library, not a full DB | Building a custom search engine |

See `real_vector_dbs.py` in this folder for example code for each one.

## Cell 6: Practical example (embed docs, store, search)

Let's put it all together: Day 1's documents, Day 2's embedding approach, and
today's vector database.

In [ ]:
documents = [
    {"title": "What is RAG",
     "text": "RAG stands for Retrieval-Augmented Generation. It is a technique where a system retrieves relevant documents before generating an answer."},
    {"title": "What is a Vector Database",
     "text": "A vector database stores data as numerical vectors called embeddings. It allows fast similarity search."},
    {"title": "Python Data Types",
     "text": "Python has several built-in data types including strings, integers, floats, lists, and dictionaries."},
]

CONCEPTS = {
    "retrieval": ["rag", "retrieval", "retrieves", "generation", "lookup", "look", "find"],
    "search_tech": ["vector", "database", "embeddings", "similarity", "search", "meaning"],
    "data_types": ["data", "types", "strings", "integers", "floats", "lists", "dictionaries"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!") for w in text.lower().split())
    return [float(sum(1 for w in words if w in concept_words)) for concept_words in CONCEPTS.values()]

doc_db = SimpleVectorDB()
for doc in documents:
    embedding = text_to_embedding(doc["title"] + " " + doc["text"])
    doc_db.add(embedding, metadata={"title": doc["title"]})

query = "How do I search by meaning?"
query_embedding = text_to_embedding(query)
results = doc_db.search(query_embedding, top_k=3)

print(f"Query: {query}\n")
for r in results:
    print(f"  {r['metadata']['title']:28s} similarity={r['similarity']:.3f}")

## Cell 7: Key takeaways

- A vector database stores embeddings and finds the closest ones to a query, fast.
- Brute force search checks everything, and gets slower as data grows.
- Indexing (like HNSW) trades a small amount of accuracy for a big speed gain, by
  only checking a small, well-chosen slice of the data.
- Real vector databases (Pinecone, Milvus, Weaviate, Chroma, FAISS) do this at
  production scale, with persistence, filtering, and reliability built in.
- This is the missing piece that makes RAG practical with real amounts of data --
  without it, every question would mean scanning your entire document collection
  from scratch.